# Letting scqubits choose your multiprocessing settings

J. Koch and P. Groszkowski

For further documentation of scqubits see https://scqubits.readthedocs.io/en/latest/.

---

A `ParameterSweep` can spread its grid points across worker processes through the `num_cpus`
argument. The catch is that the *right* number of workers — and the right number of BLAS
threads each worker should use — depends on the problem and the machine. Guess wrong and you
get no speedup, or, when the workers oversubscribe the cores, a slowdown of **one to two
orders of magnitude**.

So scqubits can choose for you. This notebook shows the easy path —
`recommend_parallelization` and `num_cpus="auto"` — and a one-time per-machine calibration,
and then explains what is being balanced under the hood.

In [ ]:
import numpy as np

import scqubits as scq

## A system to sweep

We use three capacitively coupled tunable transmons and sweep the flux of the first. The
dressed Hilbert space has dimension `6**3 = 216`.

In [ ]:
def build_hilbertspace():
    qubits = [
        scq.TunableTransmon(
            EJmax=30.0, EC=0.2, d=0.1, flux=0.0, ng=0.0, ncut=50,
            truncated_dim=6, id_str=f"tmon{i}",
        )
        for i in range(3)
    ]
    hs = scq.HilbertSpace(qubits)
    for i in range(2):
        hs.add_interaction(
            g_strength=0.1, op1=qubits[i].n_operator, op2=qubits[i + 1].n_operator
        )

    def update(flux):
        qubits[0].flux = flux

    return hs, update


hs, update = build_hilbertspace()
print("dressed dimension:", hs.dimension)

## The easy way: let scqubits choose

`scq.recommend_parallelization` reads the workload — Hilbert-space dimension, number of grid
points, eigenvalue count, and whether sparse diagonalization applies — and returns a
recommended `num_cpus` together with a per-worker BLAS-thread cap. It is a *pure* function: it
starts no worker processes, so it is safe to call anywhere, and it does not run the sweep.

Because a `ParameterSweep` runs the moment it is constructed, call it *before* building the
sweep. Here is its choice for our system at two grid sizes:

In [ ]:
for n_points in (16, 384):
    cfg = scq.recommend_parallelization(
        hilbertspace=hs, num_points=n_points, evals_count=20
    )
    print(f"{n_points:>4} points -> num_cpus={cfg.num_cpus}, blas_threads={cfg.blas_threads}")
    print(f"            {cfg.reason}")

A 16-point sweep stays serial — there are too few points to repay the cost of starting and
feeding worker processes — while the 384-point sweep is spread across workers, each capped to
a single BLAS thread so they do not oversubscribe the cores.

To apply the recommendation without copying numbers by hand, pass the sentinel
`num_cpus="auto"` to the sweep. The choice is then made automatically, *before* the sweep
runs:

```python
sweep = scq.ParameterSweep(..., num_cpus="auto")
```

To make *every* sweep that does not specify `num_cpus` tune itself this way, set
`scq.settings.AUTO_PARALLEL = True`.

These are the same auto-tuner with different reach — `num_cpus="auto"` opts in for one sweep, while `AUTO_PARALLEL = True` makes it the default for every sweep where you don't pass `num_cpus`. An explicit number always wins:

```text
num_cpus=4        ->  exactly 4 workers      (you decide)
num_cpus="auto"   ->  auto-tuner decides     (always)
num_cpus omitted  ->  auto-tuner if AUTO_PARALLEL=True, else serial (the default)
```

The automatic choice changes only *how* a sweep is computed, never the result. We confirm
that by running the same 384-point sweep both serially and with `num_cpus="auto"`, and
comparing the spectra:

In [ ]:
flux_vals = np.linspace(0.0, 0.5, 384)

serial = scq.ParameterSweep(
    hilbertspace=hs, paramvals_by_name={"flux": flux_vals},
    update_hilbertspace=update, evals_count=20, num_cpus=1,
)
auto = scq.ParameterSweep(
    hilbertspace=hs, paramvals_by_name={"flux": flux_vals},
    update_hilbertspace=update, evals_count=20, num_cpus="auto",
)
print("spectra identical:", np.allclose(serial["evals"][:], auto["evals"][:]))

## Tune to your machine (optional, run once)

The recommendation above uses conservative built-in thresholds that work everywhere. For choices tuned to *your* hardware, run the one-time calibration. It times a short battery of sweeps in isolated subprocesses — this machine's per-task dispatch overhead, the one-time pool-startup cost, and per-point diagonalization cost (dense and sparse) — and writes `~/.scqubits/parallel_calibration.json` (about a minute).

The calibration is **just data**: it does nothing on its own. From then on, every `recommend_parallelization` / `num_cpus="auto"` call reads that file and makes a sharper, machine-specific choice instead of using the generic defaults.

**Re-running overwrites the file, so recalibrate freely** — and *do* recalibrate if a previous run was taken under bad conditions: while the machine was busy, or on a laptop that was on battery / CPU-throttled (many laptops clock down hard when unplugged, which makes the calibration over-estimate every cost so `"auto"` then under-parallelizes). For the most representative numbers, calibrate on an otherwise-idle machine plugged into wall power.

Because it launches its measurements as `python -m` subprocesses, the call needs no `if __name__ == "__main__":` guard, in Jupyter or in a plain script.

In [ ]:
scq.calibrate_parallelization()

Once the calibration exists, `recommend_parallelization` (and `num_cpus="auto"`) use the
measured break-even — parallelizing only once the grid is large enough to repay the measured
pool-startup cost. Re-running the recommendation now reflects your machine:

In [ ]:
for n_points in (16, 384):
    cfg = scq.recommend_parallelization(
        hilbertspace=hs, num_points=n_points, evals_count=20
    )
    print(f"{n_points:>4} points -> num_cpus={cfg.num_cpus}  ({cfg.reason})")

## What scqubits is balancing for you

There is no single right answer because two effects pull in opposite directions.

**1. The grid break-even.** Sending a grid point to a worker costs a fixed amount (pickling,
inter-process hand-off), and starting the pool costs a one-time amount (about a second when
workers are *spawned*, as on macOS and Windows). Parallelism pays off only when

> (number of grid points) × (cost per point) ≫ that fixed overhead.

Few points, or cheap points, stay faster serially — which is why the heuristic keeps small
sweeps on a single process.

**2. The BLAS oversubscription cliff.** Every eigensolve already runs on a multithreaded BLAS
backend. If several workers each use all cores, the cores are oversubscribed by a factor of
`num_cpus`, which on large dense matrices is not a small slowdown but a collapse. Measured on
a 10-core Mac mini — five capacitively coupled fluxonia (dressed dimension 3125, dense),
16-point sweep:

| configuration | wall time |
|---|---:|
| `num_cpus=1` | 42 s |
| `num_cpus=4`, BLAS uncapped | **3608 s**  (~90x slower) |
| `num_cpus=4`, BLAS capped to 1 | 40 s |
| `num_cpus=8`, BLAS capped to 1 | 28 s |

This is why the recommendation always pairs a worker count with a BLAS-thread cap, keeping
`num_cpus x BLAS-threads` near the core count.

## Manual control

The same knobs are available directly if you would rather set them yourself:

```python
scq.settings.NUM_CPUS = 4                 # default worker count when num_cpus is unset
scq.settings.MULTIPROC_BLAS_THREADS = 1   # per-worker BLAS-thread cap during a sweep
```

`MULTIPROC_BLAS_THREADS` already defaults to `"auto"`, which caps each worker to
`cores // num_cpus` so parallel sweeps never oversubscribe the cores — setting an integer
just overrides that with a fixed cap (use `None` to opt out entirely). Rule of thumb: keep
`num_cpus x BLAS-threads` near the number of physical cores. The cap reaches spawn-based
workers (macOS, Windows) through the thread-count environment variables, and fork-based
workers (Linux) through `threadpoolctl`.

For large composite systems the per-point **diagonalization method** is often a bigger lever
than parallelism: scqubits uses sparse diagonalization by default for large spectra (see
`scq.settings.AUTO_SPARSE_DIAG`), which can be far faster per point. Once each point is cheap,
parallelism helps even less — so try sparse first, and parallelize second.

## Running as a script

In Jupyter, everything above runs as shown. In a plain Python script on macOS or Windows,
workers are *spawned*, which re-imports your script in each worker — so the entry point that
triggers a parallel sweep must be guarded:

```python
import scqubits as scq

if __name__ == "__main__":
    sweep = scq.ParameterSweep(..., num_cpus="auto")
```

Linux (which forks) and Jupyter need no guard. scqubits prints a one-time reminder the first
time it spawns workers outside of IPython.

## Summary

- **Let scqubits choose.** `scq.recommend_parallelization(...)` recommends `num_cpus` and a
  BLAS-thread cap from the workload; `num_cpus="auto"` applies it per sweep, and
  `scq.settings.AUTO_PARALLEL = True` applies it everywhere.
- **Calibrate once** with `scq.calibrate_parallelization()` for advice measured on your own
  hardware.
- Parallelism helps only when the grid is large enough to repay the fixed overhead, and the
  BLAS-thread cap is what keeps many workers from oversubscribing the cores.
- All of this takes effect live, without restarting the kernel, in Jupyter and in scripts
  alike.